# 08 · Food Relief Priority Index

Combine P(High ∪ Severe) with a food-fragility score. Fragility scalers are fit on TRAIN only, then applied to the test set. Maps rendered with folium for Ida (LA) and Ian (FL).

In [ ]:
import sys, os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', 60)
from config import (
    DATA_PATHS, HURRICANE_META, STATES_IN_SCOPE,
    TARGET_COL, TARGET_CLASS_COL, FEATURE_GROUPS,
    RANDOM_STATE, SEVERITY_BINS, SEVERITY_LABELS,
)
RAW = DATA_PATHS['raw']; INTERIM = DATA_PATHS['interim']
PROC = DATA_PATHS['processed']; MODELS = DATA_PATHS['models']
OUT = DATA_PATHS['outputs']


In [ ]:
import joblib, folium
from src.priority_index import fit_fragility_scalers, apply_fragility, priority_index, save_scalers
df = pd.read_csv(PROC / 'abt_with_clusters.csv', dtype={'zip_code': str}).dropna(subset=[TARGET_CLASS_COL])
pkg = joblib.load(MODELS / 'best_classifier.pkl')
pipe, le, features = pkg['pipe'], pkg['label_encoder'], pkg['features']

In [ ]:
# Fit fragility scalers on TRAIN only
train = df[df['train_test_split']=='TRAIN']
scalers = fit_fragility_scalers(train)
save_scalers(scalers, MODELS / 'fragility_scalers.pkl')
df = apply_fragility(df, scalers)

In [ ]:
# Predict probabilities for TEST set
te = df[df['train_test_split']=='TEST'].copy()
proba = pipe.predict_proba(te[features])
te_prio = priority_index(te, proba, list(le.classes_))
top = te_prio.sort_values('priority_rank').head(20)
top[['priority_rank','zip_code','state','hurricane_name',
     'food_relief_priority_index','prob_high_or_severe',
     'food_fragility_score','svi_overall','damage_severity_class']]

In [ ]:
te_prio.to_csv(OUT / 'priority_rankings.csv', index=False)
print('saved priority_rankings.csv', te_prio.shape)

## Folium choropleth maps — Ida and Ian

In [ ]:
import geopandas as gpd
zcta_path = next((RAW / 'zcta').rglob('*.shp'))
zcta = gpd.read_file(zcta_path).to_crs('EPSG:4326')
zcta['zip_code'] = zcta['ZCTA5CE20'].astype(str).str.zfill(5)
for hn, center in [('Ida', (30.2, -90.9)), ('Ian', (26.6, -81.9))]:
    sub = te_prio[te_prio['hurricane_name']==hn]
    z = zcta.merge(sub[['zip_code','priority_index_norm']], on='zip_code')
    if z.empty:
        print('no zips for', hn); continue
    m = folium.Map(location=center, zoom_start=7)
    folium.Choropleth(
        geo_data=z.__geo_interface__, data=sub,
        columns=['zip_code','priority_index_norm'],
        key_on='feature.properties.zip_code',
        fill_color='YlOrRd', legend_name=f'Priority index — {hn}').add_to(m)
    for _, r in z.iterrows():
        folium.GeoJson(r.geometry,
            tooltip=f"ZIP {r['zip_code']}  idx={r['priority_index_norm']:.2f}").add_to(m)
    out = OUT / f'priority_map_{hn.lower()}.html'
    m.save(str(out)); print('saved', out)

## Top-50 vs bottom-50 summary

In [ ]:
top50 = te_prio.nsmallest(50, 'priority_rank')
bot50 = te_prio.nlargest(50, 'priority_rank')
pd.DataFrame({
  'top50': [top50['food_desert_flag'].sum(), top50['svi_overall'].mean(),
            top50[TARGET_COL].mean()],
  'bot50': [bot50['food_desert_flag'].sum(), bot50['svi_overall'].mean(),
            bot50[TARGET_COL].mean()],
}, index=['food_desert_count','mean_svi','mean_damage_per_1k'])